# prep_06_education
## Ohio Dental Clinic — Site Selection Analysis

**Purpose:** Processes ACS 5-Year Estimates table B15003 (Educational Attainment for Population 25+). Calculates bachelor's degree attainment rate for all 1,233 Ohio ZCTAs. Education is reported descriptively only — excluded from scoring due to near-redundancy with income (Pearson r = 0.89).

| | |
|---|---|
| **Input** | `B15003_education_raw.csv` (ACS 2020–2024) |
| **Output** | `B15003_education_cleaned.csv` |
| **Records** | 1,233 Ohio ZCTAs |

In [1]:
# Import libraries
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings("ignore")

# PATHS 
RAW_DATA_PATH = r"C:\Users\mosun\Downloads\oh_clinic_rw_files"
OUTPUT_PATH = r"../data/cleaned"

# LOAD
print("Loading B15003_education_raw.csv...")
df = pd.read_csv(
    os.path.join(RAW_DATA_PATH, "ACSDT5Y2024.B15003-Data.csv"),
    header=0,       # row 1 = column codes → used as column names
    skiprows=[1],   # row 2 = human labels → skip it
    dtype=str,
    low_memory=False
)
print(f"Raw file: {len(df)} rows | {len(df.columns)} columns")

# Extract 5-digit ZIP from NAME column ("ZCTA5 43001" → "43001")
df["zip"] = df["NAME"].str.extract(r"(\d{5})")
df = df.dropna(subset=["zip"])
print(f"After ZIP extraction: {len(df)} rows")

def to_num(col):
    """Convert column to numeric, replacing Census suppression codes with 0."""
    s = pd.to_numeric(df[col], errors="coerce")
    s = s.replace(-666666666, np.nan).replace(-999999999, np.nan)
    return s.fillna(0)

# Total adults 25+
df["edu_total"] = to_num("B15003_001E")

# Less than high school: no schooling through 12th grade no diploma
# Columns 002 through 016
df["less_than_hs"] = sum(
    to_num(f"B15003_{str(i).zfill(3)}E") for i in range(2, 17)
)

# High school diploma or GED
df["hs_or_ged"] = (
    to_num("B15003_017E") +   # Regular HS diploma
    to_num("B15003_018E")     # GED or alternative credential
)

# Some college or associate's degree
df["some_college_assoc"] = (
    to_num("B15003_019E") +   # Some college, less than 1 year
    to_num("B15003_020E") +   # Some college, 1+ years, no degree
    to_num("B15003_021E")     # Associate's degree
)

# Bachelor's degree and above
df["bachelor_plus"] = (
    to_num("B15003_022E") +   # Bachelor's degree
    to_num("B15003_023E") +   # Master's degree
    to_num("B15003_024E") +   # Professional school degree
    to_num("B15003_025E")     # Doctorate degree
)

# Percentage with bachelor's or above — key metric for the analysis
df["pct_bachelor_plus"] = (
    df["bachelor_plus"] / df["edu_total"].replace(0, np.nan) * 100
).round(1)

# Sanity check — groups should sum to total
df["_group_sum"] = (
    df["less_than_hs"] + df["hs_or_ged"] +
    df["some_college_assoc"] + df["bachelor_plus"]
)
df["_diff"] = (df["edu_total"] - df["_group_sum"]).abs()
bad = df[df["_diff"] > 50]
if len(bad) > 0:
    print(f"Warning: {len(bad)} ZIPs have group sum differing from total by >50")
else:
    print(f"Sanity check passed — education groups sum matches total for all ZIPs")

# Report
print(f"ZIPs with missing education data: {df['edu_total'].isna().sum()}")
print(f"Average bachelor's+ rate across Ohio ZIPs: {df['pct_bachelor_plus'].mean():.1f}%")

# Keep final columns
edu_clean = df[[
    "zip", "edu_total",
    "less_than_hs", "hs_or_ged", "some_college_assoc",
    "bachelor_plus", "pct_bachelor_plus"
]].copy()

# SAVE
edu_clean.to_csv(os.path.join(OUTPUT_PATH, "B15003_education_cleaned.csv"), index=False)
print(f"\nSaved: B15003_education_cleaned.csv")
print(f"Rows: {len(edu_clean)} | Columns: {list(edu_clean.columns)}")
print(f"\nSample:")
print(edu_clean.head(5).to_string(index=False))

Loading B15003_education_raw.csv...
Raw file: 1233 rows | 53 columns
After ZIP extraction: 1233 rows
Sanity check passed — education groups sum matches total for all ZIPs
ZIPs with missing education data: 0
Average bachelor's+ rate across Ohio ZIPs: 23.1%

Saved: B15003_education_cleaned.csv
Rows: 1233 | Columns: ['zip', 'edu_total', 'less_than_hs', 'hs_or_ged', 'some_college_assoc', 'bachelor_plus', 'pct_bachelor_plus']

Sample:
  zip  edu_total  less_than_hs  hs_or_ged  some_college_assoc  bachelor_plus  pct_bachelor_plus
43001       2152            43        648                 701            760               35.3
43002        909            44         67                 280            518               57.0
43003       2803           197       1261                 766            579               20.7
43004      20870          2709       4211                4592           9358               44.8
43005        195            77        106                  12              0          